** Using the feature layers to create tiling index, vector tile package and published in ArcGIS Portal **

In [1]:
import os
import time
import re
import arcpy
import traceback
from arcgis.gis import GIS

** Update the Parameter base upon the data **

In [2]:
# --- Configuration Setup ---
CONFIG = {
    "utility": "BELLONTARIO",
    "map_type": "NETWORK",
    "wkid": 3857,
    # Clean path definition
    "package_dir": r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK", 
    
    # Grouping Scales for easier access
    "scales": {
        "default":   {"min": 0, "max": 0},
        "target_30k": {"min": 50000,  "max": 0},
        "target_10k": {"min": 10000,  "max": 0}
    },
    
    # Using SETS {} for instant lookup
    # Stored as UPPERCASE to ensure case-insensitive matching in your logic
    "layers": {
        "skip": {
            "WORLD TOPOGRAPHIC MAP", 
            "WORLD HILLSHADE", 
            "BASEMAP"
        },
        "preserve_scale": {
            "VAULT", "SERVICE_BOX","POLE",
            "MANHOLE", "HANDHOLE","FIBER_SPLICE_CASE",
            "FIBER_SPITTER", "FIBER_PROTECTION","FIBER_DISTRIBUTION_HUB","FIBER_CABINET", "PULL_BOX", "FIBER_BURIED_RAP", 
            "FIBER_AERIAL_RAP", "COPPER_SITE", "COPPER_REPEATER", "COPPER_LOADCOIL", "COPPER_CROSS", "COPPER_BURIED_RAP",
            "COPPER_AERIAL_RAP", "BIP", "ANCHOR", "LLD", "PLACE_NAMES", "BUILDINGS_TEXT", "ADDRESS", "WATER", "RAILWAY",
            "PARCELS", "STREET_MISC", "BUILDINGS", "STREETS"
        },
         "default": {
            "TRENCH", "DUCT"
        },
        "scale_30k": {
            "SERVICES","OH_SECONDARY", "UGNOTES"
        },
        "scale_10k": {
            "TRANSFORMER", "SUBSTATION", "STRUCTURE"
        }
    }
}

# --- CONFIGURATION ---
# Set scales here for Vector Tile Package layer
MIN_CACHED_SCALE = 144448 # 1155581, 577791, 288895 or 144448
MAX_CACHED_SCALE = 282 # 564, 282 or 141

print(f"Parameter set for {CONFIG['utility']} {CONFIG['map_type']} with WKID {CONFIG['wkid']}")

Parameter set for BELLWEST LANDBASE with WKID 3857


** Function to set the scale of layers **

In [26]:
def set_layer_scale(layer_list, config):
    """
    Applies scale ranges based on the provided configuration dictionary.
    """
    if not layer_list:
        return

    print(f"Processing scale for {len(layer_list)} layers...")

    for layer in layer_list:
        try:
            # Normalize name to UPPERCASE for robust matching
            # (Matches the keys we stored in CONFIG["layers"])
            lyr_name = layer.name.upper()

            # Check Skip List first (Fast O(1) Lookup)
            if lyr_name in config["layers"]["skip"]:
                # Optional feature: Auto-turn off basemaps if they are on
                if layer.visible: 
                    print(f"Skipping & Hiding Base Map: {layer.name}")
                    layer.visible = False 
                else:
                    print(f"Skipping: {layer.name}")
                continue 
                
            if lyr_name in config["layers"].get("preserve_scale", []):
                print(f"Keeping Feature scale for: {layer.name}")
                continue

            # Check if layer actually supports scaling
            if not (layer.supports("minThreshold") and layer.supports("maxThreshold")):
                # print(f"Skipping (not supported): {layer.name}")
                continue

            # Target Scale using Dictionary Lookup           
            if lyr_name in config["layers"]["scale_30k"]:
                target_min = config["scales"]["target_30k"]["min"]
                target_max = config["scales"]["target_30k"]["max"]
                scale_desc = "30k Specific"
                
            elif lyr_name in config["layers"]["scale_10k"]:
                target_min = config["scales"]["target_10k"]["min"]
                target_max = config["scales"]["target_10k"]["max"]
                scale_desc = "10k Specific"
                
            else:
                target_min = config["scales"]["default"]["min"]
                target_max = config["scales"]["default"]["max"]
                scale_desc = "Default"

            # Apply the Scale
            # Only print if we are actually changing it 
            if layer.minThreshold != target_min or layer.maxThreshold != target_max:
                layer.minThreshold = target_min
                layer.maxThreshold = target_max
                print(f"Updated {layer.name}: {scale_desc} (Min: {target_min})")
            
        except Exception as e:
            print(f"Error setting scale for {layer.name}: {e}")

    print("Completed running 'Set_Layer_scale function'\n")

** Function to Extract the code **

In [27]:
def extract_layer_codes(name_list):
    """
    Parses a list of layer names to extract unique codes (suffixes).
    Assumes codes are the segment after the last underscore (e.g., 'Data_Layer_ON01' -> 'ON01').

    Args:
        name_list (list): A list of strings (layer names).

    Returns:
        set: A set of unique codes found.
    """
    if not name_list:
        return set()

    # string.split('_') is faster than re.split for single characters
    # [-1] grabs the last part
    # set() automatically removes duplicates
    unique_codes = {name.split('_')[-1] for name in name_list if '_' in name}
    
    # Handle case where no underscores existed
    if not unique_codes and name_list:
        # If names didn't have underscores, decide if you want to return the whole name or nothing.
        # Currently returns nothing to be safe.
        print("Warning: No underscores found in layer names.")

    return unique_codes

print("Completed running function 'Extract Layer Codes'")

Completed running function 'Extract Layer Codes'


In [28]:
# Updated v1
import arcpy
import os

try:
    # Initialize Project
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    
    # activeMap can be None if you are looking at a Layout or Catalog pane.
    active_map = aprx.activeMap
    
    if not active_map:
        print("Active view is not a Map. Searching project for maps...")
        maps = aprx.listMaps()
        if len(maps) > 0:
            active_map = maps[0] # Default to the first map in the project
            print(f"Defaulting to first map: {active_map.name}")
        else:
            raise Exception("No maps found in the project.")
    else:
        print(f"Active Map: {active_map.name}")

    # Set Spatial Reference
    try:
        new_cs = arcpy.SpatialReference(CONFIG["wkid"]) 
        active_map.spatialReference = new_cs
        print(f"Spatial Reference set to: {active_map.spatialReference.name}")
    except Exception as e:
        print(f"Warning: Could not set Spatial Reference. {e}")

    # Process Layers
    # Check for Broken Layers ---
    all_layers = active_map.listLayers()
    
    # Create a list of ONLY valid layers (excludes broken links)
    valid_layers = [l for l in all_layers if not l.isBroken]

    if not valid_layers:
        raise Exception("No valid (unbroken) layers found in the active map.")
    
    if len(valid_layers) < len(all_layers):
        print(f"Warning: {len(all_layers) - len(valid_layers)} broken layers were skipped.")

    # Apply Scaling Rules
    try:
        set_layer_scale(valid_layers, CONFIG)
        print("----- Scale Updated -----")        
    except Exception as e:
        print(f"Warning: Issue in scaling function: {e}")

    # Rename Map based on First Valid Layer
    first_layer_name = valid_layers[0].name
    print(f"First Layer: {first_layer_name}")

    map_codes = extract_layer_codes([first_layer_name]) 
    
    if map_codes:
        new_map_name = list(map_codes)[0]
        print(f"Renaming map from '{active_map.name}' to '{new_map_name}'...")
        active_map.name = new_map_name
    else:
        print("Warning: Could not extract code from first layer. Map name unchanged.")

    # Set Extent & Zoom
    found_valid_layer = False 
    map_view = aprx.activeView
    
    print("\nSearching for data extent...")
    # Reset environments first to ensure no old settings persist
    arcpy.ClearEnvironment("extent")
    
    for layer in valid_layers:
        # Skip Group Layers and check Datasource support ---
        # Group layers do not have a standard 'datasource' and can cause extent errors
        if layer.isGroupLayer:
            continue

        if layer.supports("DATASOURCE") and layer.visible:
            try:
                # Use arcpy.Describe to get the extent (Works on all versions)
                desc = arcpy.Describe(layer)
                
                if not desc.extent or desc.extent.width == 0:
                    print(f"Skipping {layer.name}: Extent is empty.")
                    continue

                # Project the extent to match the Map's Spatial Reference
                projected_extent = desc.extent.projectAs(active_map.spatialReference)
                
                # Check if extent is valid (has width/height)
                if projected_extent.width > 0 :
                    # This stops the tool from guessing (and guessing wrong).
                    arcpy.env.extent = projected_extent
                    arcpy.env.outputCoordinateSystem = active_map.spatialReference
                    
                    print(f"---> Environment extent set to: {layer.name}")
                    
                    if map_view:
                        map_view.camera.setExtent(projected_extent)
                    
                    # print(f"Extent set from layer: {layer.name}")
                    found_valid_layer = True
                    break
                
            except Exception as e:
                # If a specific layer fails (e.g., query layer issues), print and continue to next
                print(f"Skipping layer {layer.name} due to extent error: {e}")
                continue

    if not found_valid_layer:
        print("Warning: No valid data layers found to set extent.")

    # Export Thumbnail
    if map_view:
        thumb_name = f"VTPK_{CONFIG['utility']}_{CONFIG['map_type']}_{active_map.name}.png"
        thumb_path = os.path.join(CONFIG["package_dir"], thumb_name)
        
        print(f"Exporting thumbnail to: {thumb_path}")
        
        try:
            if 'projected_extent' in locals():
                map_view.camera.setExtent(projected_extent)
            
            # PAUSE to let the rendering engine catch up
            # 2 seconds is usually enough; increase to 5 if layers are heavy
            print("Waiting for map rendering...")
            time.sleep(2) 

            # Export
            map_view.exportToPNG(thumb_path, width=600, height=400)
            
            # Check if file size is too small (indicates blank image)
            if os.path.exists(thumb_path) and os.path.getsize(thumb_path) < 5000:
                print("Warning: Thumbnail seems suspiciously small (likely blank). Increase sleep time.")
        except Exception as e:
            print(f"Warning: Could not export thumbnail (View might not be ready). {e}")

    # Set Workspace
    workspace = aprx.defaultGeodatabase
    if workspace and arcpy.Exists(workspace):
        arcpy.env.workspace = workspace
        print(f"Workspace set to: {arcpy.env.workspace}")
    else:
        print("Warning: Default geodatabase not found.")

    # Define Outputs
    output_index_features = f"VTPK_{CONFIG['utility']}_{CONFIG['map_type']}_{active_map.name}_Tile_Index"
    
    vtpk_name = f"VTPK_{CONFIG['utility']}_{CONFIG['map_type']}_{active_map.name}.vtpk"
    output_vtpk_package = os.path.join(CONFIG["package_dir"], vtpk_name)
    
    print("\n-----------------------------------")
    print(f"Ready to Process: {output_index_features}")
    print(f"Output VTPK:      {output_vtpk_package}")
    print("-----------------------------------")

except Exception as e:
    import traceback
    print(f"\nCRITICAL ERROR: {e}")
    print(traceback.format_exc()) # Prints the exact line number where the error occurred

Active Map: ABNORTH
Spatial Reference set to: WGS_1984_Web_Mercator_Auxiliary_Sphere
Processing scale for 13 layers...
Keeping Feature scale for: LLD
Keeping Feature scale for: PLACE_NAMES
Keeping Feature scale for: BUILDINGS_TEXT
Keeping Feature scale for: ADDRESS
Keeping Feature scale for: WATER
Keeping Feature scale for: RAILWAY
Keeping Feature scale for: PARCELS
Keeping Feature scale for: STREET_MISC
Keeping Feature scale for: BUILDINGS
Keeping Feature scale for: STREETS
Skipping & Hiding Base Map: World Topographic Map
Skipping: World Hillshade
Completed running 'Set_Layer_scale function'

----- Scale Updated -----
First Layer: BELLWEST_LANDBASE_ABNORTH
Renaming map from 'ABNORTH' to 'ABNORTH'...

Searching for data extent...
---> Environment extent set to: LLD
Exporting thumbnail to: C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_BELLWEST_LANDBASE_ABNORTH.png
Waiting for map rendering...
Workspace set to: C:\Users\sameer.bajracharya\Documents\ArcGIS\Package

In [32]:
try:   
    print("Disabling scale ranges to generate accurate Index Features...")
    layer_scales = {}

    basemap_keywords = ["Topographic", "Hillshade", "World Topographic", "World Hillshade"]
    
    valid_layers = []
    for lyr in active_map.listLayers():
        if lyr.isFeatureLayer and lyr.visible:
            valid_layers.append(lyr)
            if hasattr(lyr, "minScale") and hasattr(lyr, "maxScale"):
                layer_scales[lyr.name] = (lyr.minScale, lyr.maxScale)
                lyr.minScale = 0
                lyr.maxScale = 0

    if not valid_layers:
        raise Exception("No visible feature layers found. Cannot create vector tile index.")

    print(f"Found {len(valid_layers)} valid feature layers.")

    # --- Compute combined extent of all feature layers ---
    print("Computing combined extent...")
    combined_extent = None

    for lyr in valid_layers:
        desc = arcpy.Describe(lyr)
        if hasattr(desc, "extent") and desc.extent:
            ext = desc.extent

            if combined_extent is None:
                combined_extent = ext
            else:
                combined_extent = arcpy.Extent(
                    min(combined_extent.XMin, ext.XMin),
                    min(combined_extent.YMin, ext.YMin),
                    max(combined_extent.XMax, ext.XMax),
                    max(combined_extent.YMax, ext.YMax)
                )


    if combined_extent is None:
        raise Exception("Could not compute a valid extent from feature layers.")

    arcpy.env.extent = combined_extent
    print(f"Using combined extent: {combined_extent}")

    print("\n--- FEATURE LAYER DIAGNOSTIC ---")

    for lyr in active_map.listLayers():
        print(f"\nLayer: {lyr.name}")
        print(f"  isFeatureLayer: {lyr.isFeatureLayer}")
        print(f"  visible: {lyr.visible}")
    
        try:
            desc = arcpy.Describe(lyr)
            if hasattr(desc, "featureClass"):
                print(f"  Shape type: {desc.shapeType}")
            if hasattr(desc, "extent"):
                print(f"  Extent: {desc.extent}")
        except:
            print("  Describe failed")

    print("MAP LAYERS (TOP TO BOTTOM):")
    for lyr in active_map.listLayers():
        print(" -", lyr.name, "(FeatureLayer:" , lyr.isFeatureLayer, ")")

    # count = 0
    # for lyr in active_map.listLayers():
    #     if lyr.isFeatureLayer:
    #         fc = lyr.dataSource
    #         result = arcpy.management.GetCount(fc)
    #         print(lyr.name, ":", result[0])
    #         count += int(result[0])
    
    # print("\nTOTAL FEATURES:", count)

    print("MAP SPATIAL REFERENCE:", active_map.spatialReference.name)
           
    # Construct the full path for the output.
    out_features_path = os.path.join(workspace, output_index_features)
    # print(out_features_path)
    
    # --- Record the start time ---
    script_start_time = time.perf_counter()
    print(f"\nStarting to create vector tile index for the map: '{active_map.name}'...")

   # Execute the Create Vector Tile Index tool using the active map as input.
    arcpy.management.CreateVectorTileIndex(
        in_map=active_map,
        # out_featureclass=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\OW01_VTPK_1M_500_test.shp",
        out_featureclass=out_features_path,
        service_type="ONLINE",
        tiling_scheme=None,
        vertex_count=10000,
    )

    print(f"Successfully created index features at: {out_features_path}\n")

    # Remove the tile index layer from the layerlist before creating the vector tile package.
    remove_layer_name = output_index_features

    layer_to_remove = None
    
    for lyr in active_map.listLayers():
        if lyr.name == remove_layer_name:
            layer_to_remove = lyr
            break
    
    if layer_to_remove:
        active_map.removeLayer(layer_to_remove)
        print(f"Successfully removed layer from content: '{remove_layer_name}'\n")
    else:
        print(f"Error: Layer '{remove_layer_name}' not found in the map '{active_map.name}'.")

    # Creating a vector tile package using the index feature 
    print(f"Starting to create vector tile package for the map: '{active_map.name}'...")
    print(f"Current extent: {arcpy.env.extent}")

    print("Restoring scale ranges for Package creation...")
    for lyr in active_map.listLayers():
        if lyr.name in layer_scales:
            lyr.minScale = layer_scales[lyr.name][0]
            lyr.maxScale = layer_scales[lyr.name][1]
    
    arcpy.management.CreateVectorTilePackage(
        in_map=active_map,
        output_file= output_vtpk_package,
        service_type="ONLINE",
        tiling_scheme=None,
        tile_structure="INDEXED",
        min_cached_scale=144448, #1155581, 577791, 288895 or 144448
        max_cached_scale=282, #564 or 282 or 141
        # index_polygons=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\Alektra_HN01_Network_VTPK_Index_Feature.shp",
        index_polygons= out_features_path,
        summary="",
        tags=""
    )

    print(f"Successfully created Vector Tile Package at: {output_vtpk_package}")

except arcpy.ExecuteError:
    # Get and print geoprocessing error messages.
    print(arcpy.GetMessages(2))

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

finally:
    # --- 3. This block always runs (success or error) ---
    script_end_time = time.perf_counter()
    elapsed_time = script_end_time - script_start_time
    
    # Optional: Convert to minutes and seconds for readability
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print("\n---------------------------------")
    print(f"Script finished in {minutes} minutes and {seconds:.2f} seconds.")
    print(f"(Total elapsed: {elapsed_time:.2f} seconds)")

Disabling scale ranges to generate accurate Index Features...
Found 10 valid feature layers.
Computing combined extent...
Using combined extent: -120.282633 48.997513 -109.917396 60.000004 NaN NaN NaN NaN

--- FEATURE LAYER DIAGNOSTIC ---

Layer: BELLWEST_LANDBASE_ABNORTH
  isFeatureLayer: False
  visible: True

Layer: LLD
  isFeatureLayer: True
  visible: True
  Shape type: Point
  Extent: -119.354488929 51.8708223180001 -110.010945304 57.333598276 NaN NaN NaN NaN

Layer: PLACE_NAMES
  isFeatureLayer: True
  visible: True
  Shape type: Point
  Extent: -119.950413 49.0007150000001 -110.009368 59.8669370000001 NaN NaN NaN NaN

Layer: BUILDINGS_TEXT
  isFeatureLayer: True
  visible: True
  Shape type: Point
  Extent: -118.81636018 49.658686118 -109.992326389 56.7457765930001 NaN NaN NaN NaN

Layer: ADDRESS
  isFeatureLayer: True
  visible: True
  Shape type: Point
  Extent: -120.00010968 48.9995927100001 -110.00511144 57.29330808 NaN NaN NaN NaN

Layer: WATER
  isFeatureLayer: True
  vis